In [1]:
import sklearn
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import os
import textwrap
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set_theme('notebook', style='whitegrid', font_scale=1.25)

In [4]:
from sklearn.preprocessing import PolynomialFeatures

In [2]:
x_train_df = pd.read_csv(os.path.join('data_readinglevel', 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join('data_readinglevel', 'y_train.csv'))
x_test_df = pd.read_csv(os.path.join('data_readinglevel', 'x_test.csv'))

In [3]:
cols = [
    'avg_word_length',              # or info_characters_per_word, pick one
    'avg_sentence_length',          # or info_words_per_sentence, pick one
    'type_token_ratio',             # or info_type_token_ratio, pick one
    'pronoun_freq',
    'punctuation_frequency',
    'sentiment_polarity',
    'sentiment_subjectivity',
    'info_syll_per_word',
    'readability_Kincaid',
    'readability_ARI',
    'readability_Coleman-Liau',
    'readability_FleschReadingEase',
    'readability_GunningFogIndex',
    'readability_LIX',
    'readability_SMOGIndex',
    'readability_RIX',
    'readability_DaleChallIndex',
]
x_train_df_sliced = x_train_df[cols]
x_test_df_sliced = x_test_df[cols]
# print("X_train_df_sliced columns:")
# print(x_train_df_sliced.columns)
# print(f"X_train_df.columns: {x_train_df.columns}")

In [16]:
# load embeddings
train_embeddings = np.load('data_readinglevel\\x_train_BERT_embeddings.npz')['arr_0']
test_embeddings = np.load('data_readinglevel\\x_test_BERT_embeddings.npz')['arr_0']

print(train_embeddings.shape)  # should be (n_train_samples, 768)

# combine BERT embeddings with your numeric features
x_train_combined = np.hstack([
    train_embeddings,
    x_train_df[cols].values  # your handcrafted cols
])
x_test_combined = np.hstack([
    test_embeddings,
    x_test_df[cols].values
])

(5557, 768)


In [ ]:
def make_random_forest_pipeline_poly(max_depth=5, min_samples_leaf=2, n_estimators=200):
    steps = [
        ('rescaler', sklearn.preprocessing.MinMaxScaler()),
        ("clf", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ]
    pipeline = sklearn.pipeline.Pipeline(steps=steps)
    return pipeline

In [17]:
def make_logit_pipeline(C=1.0):
    steps = [
        ('min_max_scaler',sklearn.preprocessing.MinMaxScaler()),
        ('logit', sklearn.linear_model.LogisticRegression(solver="lbfgs", l1_ratio=0, C=C, max_iter=1000))]
    pipeline = sklearn.pipeline.Pipeline(steps=steps)
    return pipeline

In [22]:
groups = x_train_df['author']
y = y_train_df['Coarse Label']

kf = sklearn.model_selection.StratifiedGroupKFold(n_splits=10)

pl = make_logit_pipeline(C=0.01)
aucs = []
for train_index, val_index in kf.split(x_train_combined, y, groups=groups):
    x_train_fold = x_train_combined[train_index]  # numpy indexing now
    y_train_fold = y.iloc[train_index]
    x_val_fold = x_train_combined[val_index]
    y_val_fold = y.iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)

mean_auc = np.mean(aucs)
print(f"BERT + Logistic Regression AUC: {mean_auc:.4f}")

BERT + Logistic Regression AUC: 0.7727


In [23]:
pl_poly = make_logit_pipeline(C=0.01)
pl_poly.fit(x_train_combined, y_train_df['Coarse Label'])
y_test_pred_proba = pl_poly.predict_proba(x_test_combined)[:, 1]
with open("yproba1_test_p2_best_deg.txt", "w") as f:
    for pred in y_test_pred_proba:
        f.write(f"{pred}\n")

In [ ]:
C_grid = np.logspace(-4, 4, 17)
hypers_list = []

kf = sklearn.model_selection.KFold(n_splits=10, shuffle=True, random_state=42)
for C in C_grid:
  pl = make_random_forest_pipeline_poly(degree=2, C=C)
  aucs = []
  for train_index, val_index in kf.split(x_train_df_sliced):
    x_train_fold = x_train_df_sliced.iloc[train_index]
    y_train_fold = y_train_df['Coarse Label'].iloc[train_index]
    x_val_fold = x_train_df_sliced.iloc[val_index]
    y_val_fold = y_train_df['Coarse Label'].iloc[val_index]

    pl.fit(x_train_fold, y_train_fold)
    y_pred_proba = pl.predict_proba(x_val_fold)[:, 1]

    auc = sklearn.metrics.roc_auc_score(y_val_fold, y_pred_proba)
    aucs.append(auc)
  mean_auc = np.mean(aucs)
  hypers_list.append((C, mean_auc))

AUC_vals = [h[1] for h in hypers_list]
for C, AUC in zip(C_grid, AUC_vals):
    print(f"AUC of: {AUC} for C of: {C}")
plt.plot(C_grid, AUC_vals, color='red')
plt.xlabel('C')
plt.ylabel('AUCROC')
plt.title('Grid Search of C for AUCROC')

best_C, best_auc = max(hypers_list, key=lambda x: x[1])
print(f"Best C: {best_C}, Best AUC: {best_auc}")

In [ ]:
# check how many outliers exist
print((x_train_df['readability_Kincaid'] > 50).sum())
print((x_test_df['readability_Kincaid'] > 50).sum())